# **Dataset and Project Overview**

# **Walmart Recruiting - Store Sales Forecasting**
The dataset contains historical sales data for 45 Walmart stores located in different regions. Each store contains a number of departments, and the goal is to predict the department-wide sales for each store.

In addition, Walmart runs several promotional markdown events throughout the year. These markdowns precede prominent holidays, the four largest of which are the Super Bowl, Labor Day, Thanksgiving, and Christmas. The weeks including these holidays are weighted five times higher in the evaluation than non-holiday weeks. Part of the challenge presented by this competition is modeling the effects of markdowns on these holiday weeks in the absence of complete/ideal historical data.
### **Data Files**
**stores.csv**

This file contains anonymized information about the 45 stores, indicating the type and size of store.

**train.csv**

This is the historical training data, which covers to 2010-02-05 to 2012-11-01. Within this file you will find the following fields:

`Store` - the store number <br>
`Dept` - the department number <br>
`Date` - the week <br>
`Weekly_Sales` -  sales for the given department in the given store <br>
`IsHoliday` - whether the week is a special holiday week <br>

**test.csv**

This file is identical to train.csv, except we have withheld the weekly sales. You must predict the sales for each triplet of store, department, and date in this file.

**features.csv**

This file contains additional data related to the store, department, and regional activity for the given dates. It contains the following fields:

`Store` - the store number <br>
`Date` - the week <br>
`Temperature` - average temperature in the region <br>
`Fuel_Price` - cost of fuel in the region <br>
`MarkDown1-5` - anonymized data related to promotional markdowns that Walmart is running. MarkDown data is only available after Nov 2011, and is not available for all stores all the time. Any missing value is marked with an NA. <br>
`CPI` - the consumer price index <br>
`Unemployment` - the unemployment rate <br>
`IsHoliday` - whether the week is a special holiday week <br>

For convenience, the four holidays fall within the following weeks in the dataset (not all holidays are in the data):
- Super Bowl: 12-Feb-10, 11-Feb-11, 10-Feb-12, 8-Feb-13
- Labor Day: 10-Sep-10, 9-Sep-11, 7-Sep-12, 6-Sep-13
- Thanksgiving: 26-Nov-10, 25-Nov-11, 23-Nov-12, 29-Nov-13
- Christmas: 31-Dec-10, 30-Dec-11, 28-Dec-12, 27-Dec-13

The task is to predict Weekly Sales for specific departments across 45 different Walmart stores. A key challenge is that Walmart holds major promotional events (Markdowns) before four major holidays: Super Bowl, Labor Day, Thanksgiving, and Christmas. These weeks have a significantly higher impact on total revenue and are weighted more heavily in the evaluation.

### Libraries and Data

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import xgboost as xgb

from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

!pip install pmdarima
from pmdarima.arima import auto_arima

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 8.3 MB/s eta 0:00:00


!pip install pmdarima


In [ ]:
features = pd.read_csv('/content/sample_data/features.csv')
stores = pd.read_csv('/content/sample_data/stores.csv')
test = pd.read_csv('/content/sample_data/test.csv')
train = pd.read_csv('/content/sample_data/train.csv')

### Data Check

In [ ]:
print('First 5 rows of Features Dataset:')
display(features.head())
print('\nFirst 5 rows of Stores Dataset:')
display(stores.head())
print('\nFirst 5 rows of Test Dataset:')
display(test.head())
print('\nFirst 5 rows of Train Dataset:')
display(train.head())

In [ ]:
print('Shape Features Dataset:')
display(features.shape)
print('Shape Stores Dataset:')
display(stores.shape)
print('Shape Test Dataset:')
display(test.shape)
print('Shape Train Dataset:')
display(train.shape)

In [ ]:
features.info()

####
---
## Stage 1 *(The Master Merge)*
---
####

In [ ]:
# Identification of Train VS Test dataset
train['is_train'] = 1
test['is_train'] = 0

df = pd.concat([train, test], axis=0).reset_index(drop=True)
df.info()

In [ ]:
df = pd.merge(df, stores, on='Store', how='left')

print(df.shape)
display(df.head())

In [ ]:
df = pd.merge(df, features, on=['Store', 'Date', 'IsHoliday'], how='left')

print(df.shape)
display(df.head())

####
---
### Stage 2 *(Handling Missing Values)*
---
####

In [ ]:
df.info()

**Missing values appears in all Markdown 1-5, let's fill them up with 0**

In [ ]:
df.fillna(0, inplace=True)

In [ ]:
# Confirm there's no other missing values

df.isna().sum()

In [ ]:
df.info()

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.head()

####
---
### Stage 3 *(Feature Engineering)*
---
####

In [ ]:
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

df.info()

**Temporal Features** <br> Extraction of `Month`, `Day`, `WeekOfYear`, and `Year` from the Date

In [ ]:
df['Day'] = df['Date'].dt.day
df['Month'] = df['Date'].dt.month
df['Year'] = df['Date'].dt.year
df['WeekOfYear'] = df['Date'].dt.isocalendar().week

In [ ]:
df.head()

**The "Days to Holiday" Feature** <br>
Since Black Friday and Christmas move, create a counter for *"Days until next major Holiday."*

Here, I used the 4 christmas dates mentioned above in the introduction and used them as major holidays. That's what I understood by the script given

In [ ]:
major_holiday = ['31-Dec-10', '30-Dec-11', '28-Dec-12', '27-Dec-13']
major_holiday = pd.to_datetime(major_holiday)

In [ ]:
major_holiday

In [ ]:
holiday = pd.DataFrame({'next_holiday' :major_holiday})

df = df.sort_values('Date')
holiday

In [ ]:
df.dropna(subset=['Date'], inplace=True)
df = pd.merge_asof(df, holiday, left_on='Date', right_on='next_holiday', direction='forward')

df.sample(10)

In [ ]:
df['next_holiday'].unique()

In [ ]:
df['Days_To_Holiday'] = (df['next_holiday'] - df['Date']).dt.days

df.sample(10)

In [ ]:
df.info()

**Lag Features** <br> Create a column for `Sales_Last_Year` by shifting sales data by 52 weeks. This is the strongest predictor for seasonal retail

In [ ]:
df['Sales_Last_Year'] = df['Weekly_Sales'].shift(52)

df.tail()

######
Create an `Avg_Dept_Sales` column to give the model a baseline of how big a specific department usually is.
######

In [ ]:
Avg_Dept_Sales = df.groupby('Dept')['Weekly_Sales'].mean()

Avg_Dept_Sales

####
---
### Stage 4 *(Categorical Encoding)*
---
####

In [ ]:
df['Type'].unique()

In [ ]:
# Encoding `Type` column using direct mapping
mapping = {'A': 1, 'B': 2, 'C': 3}

df['Type'] = df['Type'].map(mapping)

In [ ]:
df.tail()

In [ ]:
# Converting `IsHoliday` column to Integer from Boolean
df['IsHoliday'] = df['IsHoliday'].astype(int)

display(df.head())
df.info()

####
---
### Stage 5 *(Output & Documentation)*
---
####

In [ ]:
# To reduce memory usage and file size

int_64 = df.select_dtypes(include=['int64', 'int32', 'UInt32'])

for c in int_64:
    df[c] = df[c].astype('uint8')

df.info()

In [ ]:
final_train = df[df['is_train'] == 1].drop(columns=['is_train'])
final_test = df[df['is_train'] == 0].drop(columns=['is_train', 'Weekly_Sales'])

In [ ]:
print('Final Train Dataset Shape:', final_train.shape)
print('Final Test Dataset Shape:', final_test.shape)

In [ ]:
# final_train.to_csv('train_final.csv', date_format='%Y-%m-%d', index=False)
# final_test.to_csv('test_final.csv', date_format='%Y-%m-%d', index=False)

In [ ]:
# df.to_csv('final_df.csv', date_format='%Y-%m-%d', index=False)

In [ ]:
df['Sales_Last_Year'] = df['Sales_Last_Year'].fillna(0)

# Confirm that there are no more missing values in the Sales_Last_Year column
print('Missing values in Sales_Last_Year:', df['Sales_Last_Year'].isna().sum())
display(df[['Sales_Last_Year']].head())

####
---
### Stage 6 *(Exploratory Data Analysis (EDA))*
---
####

# **Descriptive Statistics**
Weekly_Sales: mean ~13k–14k, std high (outliers in holiday weeks), min negative (returns?), max > 300k
Temperature: mean ~60°F, range -10 to 100°F
Fuel_Price: mean ~3.4, rising over time
MarkDowns: highly skewed (many 0s), mean ~1k–3k when present
CPI: mean ~170–180, Unemployment: mean ~8%
Size: mean ~130k sq ft, Type A largest

In [ ]:
# EDA: Distributions
plt.figure(figsize=(15, 10))
for i, col in enumerate(['Weekly_Sales', 'Temperature', 'Fuel_Price'], 1):
    plt.subplot(3, 1, i)
    sns.histplot(df[col], kde=True)
    plt.title(f'Distribution of {col}')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation Heatmap (numerical only)
plt.figure(figsize=(12, 8))
sns.heatmap(df.select_dtypes(include=np.number).corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
# Holiday vs Non-Holiday Sales
sns.boxplot(x='IsHoliday', y='Weekly_Sales', data=df)
plt.title('Weekly Sales: Holiday vs Non-Holiday')
plt.show()

In [ ]:
# Sales by Month
df.groupby('Month')['Weekly_Sales'].mean().plot(kind='bar')
plt.title('Average Sales by Month')
plt.show()

**6.1 More powerful features**

In [ ]:
# A. IsMajorHoliday: 1 if Super Bowl, Labor Day, Thanksgiving, Christmas week
major_holidays = df['IsHoliday'] & (
    (df['WeekOfYear'].isin([6, 36, 47, 51, 52]))  # approximate weeks for major holidays
)
df['IsMajorHoliday'] = major_holidays.astype(int)


# B. Promo Intensity: total markdown amount
df['Promo_Intensity'] = df[['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']].sum(axis=1)


# C. Store Size Bins (small/medium/large)
df['Size_Bin'] = pd.qcut(df['Size'], q=3, labels=['Small', 'Medium', 'Large'])


# D. Interaction: Size × IsHoliday
df['Size_x_Holiday'] = df['Size'] * df['IsHoliday'].astype(int)


# E. IsWeekend (if Date includes weekday info — optional)
df['IsWeekend'] = df['Date'].dt.dayofweek.isin([5, 6]).astype(int)


print("New features added:", ['IsMajorHoliday', 'Promo_Intensity', 'Size_Bin', 'Size_x_Holiday'])
df[['IsMajorHoliday', 'Promo_Intensity', 'Size_Bin', 'Size_x_Holiday']].head()

**6.2 Comprehensive Exploratory Data Analysis (EDA) & Insights (Main Focus)**

**6.2.1 Descriptive Statistics (overall & by key groups)**

In [ ]:
print("Descriptive Statistics for Key Numerical Columns:")
display(df.describe())  # Overall stats

# Grouped stats: Average sales by IsHoliday
avg_sales_holiday = df.groupby('IsHoliday')['Weekly_Sales'].mean()
print("\nAverage Weekly Sales by Holiday:")
print(avg_sales_holiday)

# Grouped stats: Average sales by Store Type
avg_sales_type = df.groupby('Type')['Weekly_Sales'].mean()
print("\nAverage Weekly Sales by Store Type:")
print(avg_sales_type)

# Grouped stats: Average sales by Month
avg_sales_month = df.groupby('Month')['Weekly_Sales'].mean()
print("\nAverage Weekly Sales by Month:")
print(avg_sales_month)

**6.2.2 Distributions (Histograms for key variables)**

In [ ]:
plt.figure(figsize=(15, 10))

plt.subplot(2, 2, 1)
sns.histplot(df['Weekly_Sales'], kde=True, bins=30)
plt.title('Distribution of Weekly Sales')
plt.xlabel('Weekly Sales')
plt.ylabel('Frequency')

plt.subplot(2, 2, 2)
sns.histplot(df['Temperature'], kde=True, bins=30)
plt.title('Distribution of Temperature')
plt.xlabel('Temperature (°F)')
plt.ylabel('Frequency')

plt.subplot(2, 2, 3)
sns.histplot(df['Promo_Intensity'], kde=True, bins=30)
plt.title('Distribution of Promo Intensity (Total Markdowns)')
plt.xlabel('Promo Intensity')
plt.ylabel('Frequency')

plt.subplot(2, 2, 4)
sns.histplot(df['Days_To_Holiday'], kde=True, bins=30)
plt.title('Distribution of Days to Next Holiday')
plt.xlabel('Days to Holiday')
plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

**6.2.3 Correlations (Heatmap for numerical features)**

In [ ]:
num_cols = ['Weekly_Sales', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment', 'Size', 'Promo_Intensity', 'Days_To_Holiday', 'Sales_Last_Year']
corr_matrix = df[num_cols].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap of Key Features')
#plt.savefig('Correlation Heatmap of Key Features', dpi=300, bbox_inches='tight')
plt.show()

**6.2.4 Groupby Visualizations & Insights**

In [ ]:
# Sales by IsHoliday (boxplot)
plt.figure(figsize=(8, 6))
sns.boxplot(x='IsHoliday', y='Weekly_Sales', data=df)
plt.title('Weekly Sales by Holiday Status')
plt.xlabel('Is Holiday')
plt.ylabel('Weekly Sales')
plt.show()

In [ ]:
# Sales by Month (bar plot)
avg_sales_month.plot(kind='bar', figsize=(10, 6))
plt.title('Average Weekly Sales by Month')
plt.xlabel('Month')
plt.ylabel('Average Weekly Sales')
plt.show()

In [ ]:
# Sales by Store Type (bar plot)
avg_sales_type.plot(kind='bar', figsize=(8, 6))
plt.title('Average Weekly Sales by Store Type')
plt.xlabel('Store Type')
plt.ylabel('Average Weekly Sales')
plt.show()

**Insights from EDA:**

*   Weekly Sales are highly skewed (long tail of high values) — likely driven by holidays. Average sales during holidays are 20–50% higher than non-holidays, with outliers exceeding $200k.


*   Seasonal patterns are strong: Sales peak in November–December (Q4, holidays), with lows in Q1 — align promotions accordingly.


*   Promo Intensity correlates moderately with sales (r~0.2–0.4), but only post-2011 data — suggest time-split modeling.


*   Larger stores (Type A) have 2x higher average sales than Type C; Size is a key predictor (r~0.25 with sales).




*   Economic factors: Unemployment negatively impacts sales (r~-0.1), while Fuel_Price has minimal effect.


*   Overall insight: Holidays and promotions are the biggest drivers — focus forecasting on these for accuracy.

**6.3 Error Analysis Prep (After Group B Provides Model)**

####
---
### Stage 7 *(Creating Pipeline)*
---
####

In [ ]:
# Filtering the training data from the merged dataframe
train_df = df[df['is_train'] == 1]

# Defining features (X) and target (y)
# Dropping the target, identification, and non-numeric datetime columns
X = train_df.drop(columns=['Weekly_Sales', 'is_train', 'Date', 'next_holiday', 'IsMajorHoliday', 'Promo_Intensity', 'Size_Bin', 'Size_x_Holiday'])
y = train_df['Weekly_Sales']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print('Features (X) shape:', X.shape)
print('Target (y) shape:', y.shape)

In [ ]:
display(X.head())

In [ ]:
# Creating the Pipeline with Scaling and XGBoost
model = Pipeline([
    ('scaler', StandardScaler()),
    ('xgb_model', xgb.XGBRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=6,
        random_state=42))
])

# Training the pipeline
model.fit(X_train, y_train)

print('Model Pipeline created and trained successfully.')

In [ ]:
model.fit(X_train, y_train)

In [ ]:
y_train_pred = model.predict(X_train)
print(y_train_pred)

In [ ]:
# Create a dataframe for Actual vs Predicted values on training dataset
train_pred_df = pd.DataFrame({'Actual': y_train, 'Predicted': y_train_pred})
display(train_pred_df.head())

In [ ]:
y_test_pred = model.predict(X_test)
print(y_test_pred)

In [ ]:
# Create dataframe of Actual vs Predicted on the test dataset
test_pred_df = pd.DataFrame({'Actual': y_test, 'Predicted': y_test_pred})
display(test_pred_df.head())

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

mae = mean_absolute_error(y_test, y_test_pred)
r2 = r2_score(y_test, y_test_pred)

print(f'Mean Absolute Error (MAE): {mae:.2f}')
print(f'R-squared Score: {r2:.4f}')

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_test, y=y_test_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], '--r', linewidth=2)
plt.xlabel('Actual Weekly Sales')
plt.ylabel('Predicted Weekly Sales')
plt.title('Actual vs Predicted Weekly Sales (Test Set)')
plt.show()

From the results on the test dataset, we can infer several key points:

1. R-squared Score (0.8970): The model explains approximately 89.7% of the variance in the weekly sales. This is a strong result, suggesting that the features (like store type, size, and the engineered lag features) are highly predictive of sales.
2. Mean Absolute Error (4267.41): On average, the model's predictions are off by about $4,267. Given that Walmart weekly sales can reach hundreds of thousands of dollars for certain departments, this level of error is relatively low, though it may be more significant for smaller departments.
3. Visualization: The scatter plot shows that the points generally cluster closely around the red diagonal line, which represents perfect predictions. However, you'll likely notice some dispersion at higher sales volumes, which is common in retail forecasting as peak holiday spikes are harder to predict perfectly.

Overall, the model is performing quite well and generalizes effectively to the unseen test data.

####
---
### Stage 8 *(Hyperparameter Tuning)*
---
####

## Define Hyperparameter Grid

### Subtask:
Create a dictionary containing the range of hyperparameters to test for the XGBRegressor within the existing pipeline.


In [ ]:
# Define the hyperparameter grid for GridSearch
# The keys must match the name of the step in the pipeline ('xgb_model') followed by double underscores
param_grid = {
    'xgb_model__max_depth': [4, 6, 8],
    'xgb_model__learning_rate': [0.05, 0.1],
    'xgb_model__n_estimators': [100, 200]
}

print('Hyperparameter grid defined:')
print(param_grid)

## Setup and Run GridSearchCV

### Subtask:
Initialize and execute a GridSearchCV to find the optimal hyperparameters for the XGBoost pipeline.


In [ ]:
from sklearn.model_selection import GridSearchCV

# Initialize GridSearchCV
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=1
)

# Fit GridSearchCV to the training data
print('Starting GridSearchCV. This may take several minutes...')
grid_search.fit(X_train, y_train)

# Store the best model found
best_model = grid_search.best_estimator_

print('GridSearchCV complete.')
print(f'Best parameters found: {grid_search.best_params_}')
print(f'Best cross-validation score (Neg MAE): {grid_search.best_score_:.2f}')

## Evaluate Tuned Model

### Subtask:
Use the best estimator found by GridSearchCV to make predictions on the test set and calculate updated R-squared and MAE metrics to compare against the baseline model.


In [ ]:
# 1. Generate predictions for the test data using the best model from GridSearchCV
y_test_pred_tuned = best_model.predict(X_test)

# 2. Calculate Mean Absolute Error (MAE)
mae_tuned = mean_absolute_error(y_test, y_test_pred_tuned)

# 3. Calculate R-squared score
r2_tuned = r2_score(y_test, y_test_pred_tuned)

# 4. Print the results
print(f'--- Tuned Model Performance ---')
print(f'Mean Absolute Error (MAE): {mae_tuned:.2f}')
print(f'R-squared Score: {r2_tuned:.4f}')

# 5. Compare with baseline performance
mae_baseline = 4267.41
r2_baseline = 0.8970

print('\n--- Comparison with Baseline ---')
print(f'MAE Improvement: {mae_baseline - mae_tuned:.2f}')
print(f'R2 Improvement: {r2_tuned - r2_baseline:.4f}')

# 6. Scatter plot of Actual vs. Predicted values for the tuned model
plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_test, y=y_test_pred_tuned, alpha=0.5, color='green', label='Tuned Model')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], '--r', linewidth=2)
plt.xlabel('Actual Weekly Sales')
plt.ylabel('Predicted Weekly Sales')
plt.title('Actual vs Predicted Weekly Sales (Tuned Model)')
plt.legend()
plt.show()

## Summary:

### Q&A
**What were the best hyperparameters found for the XGBoost model?**
The optimal configuration identified via `GridSearchCV` was:
*   `learning_rate`: 0.1
*   `max_depth`: 8
*   `n_estimators`: 200

**How did the tuned model perform compared to the baseline?**
The tuned model significantly outperformed the baseline, reducing the Mean Absolute Error (MAE) by **1,834.11** and increasing the R-squared (R²) score by **0.0674**.

### Data Analysis Key Findings
*   **Significant Error Reduction**: The Mean Absolute Error (MAE) dropped from a baseline of 4,267.41 to **2,433.30**, representing a substantial increase in the precision of weekly sales forecasts.
*   **Enhanced Variance Explanation**: The R-squared score improved from 0.8970 to **0.9644**, meaning the optimized model now accounts for approximately 96.44% of the variance in the sales data.
*   **Model Complexity**: The best parameters favored the maximum tested values for depth (8) and estimators (200), suggesting that the model benefited from higher complexity to capture the underlying patterns in the Walmart dataset.
*   **Predictive Reliability**: Visual analysis through scatter plots confirmed that the tuned model's predictions align much more closely with actual sales values across the entire distribution compared to previous iterations.


####
---
# Stage 9 *(TIme-Series Statistical Analysis)*
---
####

In the Walmart Dataset, there are 45 stores (nummbered 1-45) and each store has many departments (usually numbered 1 to 99, though not every store has every department)
Sales are recorded weekly for each Store + Department combination

This creates ~3,000–4,500 individual time series in total (one series per unique Store–Dept pair).
When using classical time series models like ARIMA / SARIMA / SARIMAX:
These models are traditionally designed to work on one single time series at a time (not thousands simultaneously).
So for the purpose of this project, the weekly sales forecasting will be done on Store 1 and Dept 1.

## **Data PreProcessing**

In [ ]:
# Prepare data for Time Series Analysis
time_series_df = train_df[(train_df['Store'] == 1) & (train_df['Dept'] == 1)].copy()

# Ensure Date is DateTime and sorted, then set as index
time_series_df['Date'] = pd.to_datetime(time_series_df['Date'])
time_series_df = time_series_df.set_index('Date').sort_index()
time_series_df = time_series_df.asfreq('W-FRI')

# For CPI and Unemployment, fill with 0 as done previously, assuming this applies to new merged data too.
time_series_df['IsHoliday'] = time_series_df['IsHoliday'].fillna(False).astype(int)
for col in ['MarkDown1','MarkDown2','MarkDown3','MarkDown4','MarkDown5', 'CPI', 'Unemployment']:
    time_series_df[col] = time_series_df[col].fillna(0)

time_series_df.drop_duplicates(inplace=True)

# Now define target and exogenous
time_series_df['log_Weekly_Sales'] = np.log1p(time_series_df['Weekly_Sales'])
Y = time_series_df['log_Weekly_Sales']

exog_cols = [
    'IsHoliday',
    'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5',
    'Temperature', 'Fuel_Price', 'CPI', 'Unemployment', 'Size'
]
exog = time_series_df[exog_cols]

# Data Splitting
train_size = int(len(time_series_df) * 0.8)

ts_train = time_series_df[:train_size]
ts_test = time_series_df[train_size:]
exog_train, exog_test = exog.iloc[:train_size], exog.iloc[train_size:]

# Scale numerical features for improved results.
from sklearn.preprocessing import StandardScaler
features_to_scale = ['Size', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment']
def scale_walmart_features(ts_train, ts_test, exog_train, exog_test, features_to_scale):
    """
    Scales numerical features using StandardScaler.
    Important: Fit on Train, Transform on both to avoid Data Leakage.
    """
    scaler = StandardScaler()

    # 1. Fit the scaler on the training data ONLY to avoid data leakage
    scaler.fit(ts_train[features_to_scale])
    scaler.fit(exog_train[features_to_scale])

    # 2. Transform both training and testing sets
    ts_train[features_to_scale] = scaler.transform(ts_train[features_to_scale])
    ts_test[features_to_scale] = scaler.transform(ts_test[features_to_scale])
    exog_train[features_to_scale] = scaler.transform(exog_train[features_to_scale])
    exog_test[features_to_scale] = scaler.transform(exog_test[features_to_scale])

    print(f"✅ Successfully scaled: {features_to_scale}")
    return ts_train, ts_test, exog_train, exog_test, scaler

# Usage:
ts_train, ts_test, exog_train, exog_test, features_to_scale = scale_walmart_features(ts_train, ts_test, exog_train, exog_test, features_to_scale)

In [ ]:
time_series_df.head()

In [ ]:
time_series_df.info()

In [ ]:
ts_train['log_Weekly_Sales'].tail()

In [ ]:
ts_test['log_Weekly_Sales'].head()

**Plot the series to show trend and seasonality**

In [ ]:
time_series_df['Weekly_Sales'].plot(figsize=(12,5))
plt.title("Weekly Sales - Store 1 Dept 1")
plt.show()

In [ ]:
time_series_df['Trend'] = time_series_df['Weekly_Sales'].rolling(window=12, center=True).mean()

plt.figure(figsize=(10,5))
plt.plot(time_series_df['Weekly_Sales'], label='Original')
plt.plot(time_series_df['Trend'], label='Trend (12-month MA)', color='red')
plt.title("Trend in Weekly Sales")
plt.xlabel("Date")
plt.ylabel("Weekly Sales")
plt.legend()
plt.show()

The target variable for this project is 'weekly_sales' which means the analysis will be done with regards to the number of weeks (52) in a year instead of months in a year (12).

In [ ]:
decomposition = seasonal_decompose(time_series_df['Weekly_Sales'], model='multiplicative', period=52)

trend = decomposition.trend
seasonal = decomposition.seasonal
residual = decomposition.resid


plt.figure(figsize=(10,8))

plt.subplot(4,1,1)
plt.plot(time_series_df['Weekly_Sales'], label='Original')
plt.legend(loc='upper left')

plt.subplot(4,1,2)
plt.plot(trend, label='Trend', color='red')
plt.legend(loc='upper left')

plt.subplot(4,1,3)
plt.plot(seasonal, label='Seasonality', color='green')
plt.legend(loc='upper left')

plt.subplot(4,1,4)
plt.plot(residual, label='Residual / Noise', color='orange')
plt.legend(loc='upper left')

plt.tight_layout()
plt.show()

##**Tests for Stationarity**

In [ ]:
result = adfuller(time_series_df['Weekly_Sales'])

print('ADF Statistic:', result[0])
print('p-value:', result[1])
if result[1] < 0.05:
    print("The series is likely stationary")
else:
    print("The series is likely non-stationary")


The target variable failed on the stationary test. To improve in this, the target variable ('Weekly_Sales') can be transformed using log transform (which has been done earlier) or differenced using the differencing method and seasonal differencing method.

In [ ]:
result = adfuller(time_series_df['log_Weekly_Sales'])

print('ADF Statistic:', result[0])
print('p-value:', result[1])
if result[1] < 0.05:
    print("The series is likely stationary")
else:
    print("The series is likely non-stationary")


The results from the Stationary Test shows that the transformed time series target variable is stationary hence, further analysis will be carried out using the transformed target variable.

### **Introduce seasonal differencing**

In [ ]:
time_series_df['Seasonal_Diff'] = time_series_df['log_Weekly_Sales'] - time_series_df['log_Weekly_Sales'].shift(52)
time_series_seasonal = time_series_df['Seasonal_Diff'].dropna()

time_series_seasonal.plot(title="Seasonally Differenced Series", figsize=(10,4))
plt.show()

In [ ]:
seasonal_result = adfuller(time_series_seasonal)

print('ADF Statistic:', seasonal_result[0])
print('p-value:', seasonal_result[1])
if result[1] < 0.05:
    print("The series is likely stationary")
else:
    print("The series is likely non-stationary")


## **AutoCorrelaction Check**

In [ ]:
plot_acf(time_series_seasonal, lags=60)
plt.show()

plot_pacf(time_series_seasonal, lags=32)
plt.show()

The results of the Autocorrelation test shows a significant spike at lag52 which strongly indicates seasonality in the target variable.

# Model Fitting - ARIMA

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(ts_train['log_Weekly_Sales'], label='Train')
plt.plot(ts_test['log_Weekly_Sales'], label='Test')
plt.legend()
plt.title("Train–Test Split (Time-Based)")
plt.show()

### **Model Fitting**

In [ ]:
model_arima = ARIMA(ts_train['log_Weekly_Sales'], order=(1, 0, 1))
fitted_model = model_arima.fit()

### **Forecasting**

In [ ]:
arima_forecast = fitted_model.forecast(steps=len(ts_test))

plt.figure(figsize=(10,4))
plt.plot(ts_train['log_Weekly_Sales'], label='Train')
plt.plot(ts_test['log_Weekly_Sales'], label='Actual')
plt.plot(arima_forecast, label='Forecast')
plt.legend()
plt.title("Forecast vs Actual")
plt.show()

# **Introducing SARIMA**

In [ ]:
model_sarimax = SARIMAX(
    ts_train['log_Weekly_Sales'],
    exog=exog_train,
    order=(1, 0, 1),
    trend='n',
    seasonal_order=(1, 1, 1, 52),
    enforce_stationarity=False,
    enforce_invertibility=False
)

sarima_results = model_sarimax.fit(method='nm', maxiter=500)

In [ ]:
forecast_obj = sarima_results.get_forecast(steps=len(ts_test['log_Weekly_Sales']), exog=exog_test)
sarimax_forecast = forecast_obj.predicted_mean
conf_int = forecast_obj.conf_int()

plt.figure(figsize=(12, 5))
plt.plot(ts_train['log_Weekly_Sales'], label='Train')
plt.plot(ts_test['log_Weekly_Sales'], label='Actual')
plt.plot(ts_test.index, sarimax_forecast, label='Forecast', color='green')
plt.fill_between(ts_test.index, conf_int.iloc[:, 0], conf_int.iloc[:, 1], color='gray', alpha=0.2)
plt.title("SARIMAX Forecast with Exogenous Variables - Store 1 Dept 1")
plt.legend()
plt.show()

**The results from the XGBoost, ARIMA and SARIMAX models are not so good but will be predicted on the Test csv nonetheless.**